# Lab Data Testing

Run the trained CNN on real lab FTIR spectra — PET and its hydrolysis products TPA (terephthalic acid) and EG (ethylene glycol) — and inspect predictions per substance.

> **Running on Kaggle:** make sure a GPU accelerator is enabled (Settings → Accelerator) and that **Internet** is turned on (Settings → Internet) so `pip install` works.

## 1. Setup

In [ ]:
!pip install -q iterative-stratification


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


## 2. Set up input paths

On Kaggle, attach two dataset inputs via **+ Add Input**:
1. A dataset with the trained model (output of `PET_FTIR_CNN_training.ipynb`), containing `weighted_ftir_cnn.keras` and `per_label_thresholds.csv`
2. A dataset with the lab data, containing `X_lab.npy` and `meta_lab.csv` from `data/lab1/` (generated from the Shimadzu JCAMP-DX files by `preprocess_lab_jcamp.py`)

> **Important:** Both `X_lab.npy` and the training data must use SNV normalization. Do not mix old min-max files with new SNV files.

Update `MODEL_DIR` and `DATA_DIR` below to match the paths shown under `/kaggle/input/` once the datasets are attached.


In [ ]:
import os

# Update these to match the dataset folders shown under /kaggle/input/
# after attaching them via "+ Add Input".
MODEL_DIR = "/kaggle/input/pet-ftir-cnn-model"
DATA_DIR  = "/kaggle/input/pet-ftir-lab1"

print("Model dir:", os.listdir(MODEL_DIR))
print("Data dir: ", os.listdir(DATA_DIR))


## 3. Load everything

In [ ]:
model       = tf.keras.models.load_model(os.path.join(MODEL_DIR, "weighted_ftir_cnn.keras"), compile=False)
X_lab       = np.load(os.path.join(DATA_DIR, "X_lab.npy"))
meta        = pd.read_csv(os.path.join(DATA_DIR, "meta_lab.csv"))
thresh_df   = pd.read_csv(os.path.join(MODEL_DIR, "per_label_thresholds.csv"))

LABEL_NAMES       = list(thresh_df["label"])
best_thresholds   = thresh_df["threshold"].values
HYDROLYSIS_NAMES  = ["ester", "carboxylic_acid", "alcohol", "arene"]

print("Model input shape:", model.input_shape)
print("Lab spectra:      ", X_lab.shape)
print("Labels:           ", LABEL_NAMES)


## 4. Predict

In [ ]:
X_input = X_lab[..., np.newaxis]
y_prob  = model.predict(X_input, verbose=0)
y_pred  = (y_prob >= best_thresholds).astype(int)
print("Predictions shape:", y_pred.shape)


## 5. Results per substance

Fraction of samples per substance where each functional group is predicted present. With only 3-4 replicates per substance, each step is 0.25-0.33 — read these as "how many of the replicates agree", not as fine-grained rates.

In [ ]:
results = pd.DataFrame(y_pred, columns=LABEL_NAMES)
results["polymer"] = meta["Polymer"].values

summary = results.groupby("polymer")[LABEL_NAMES].mean().round(3)
print(summary.to_string())
summary


## 5b. F1 score vs expected chemistry

The table above shows prediction *rates*, not accuracy. To get a proper F1 we
need ground-truth labels. The samples are known substances, so the true
functional groups follow from their molecular structure (same SMARTS
conventions as the training labels):

| Substance | True groups |
|---|---|
| PET | ester, alkane, arene, ether (ether matches the bridging ester oxygen, consistent with the SMARTS labeling of the training data) |
| TPA (terephthalic acid) | carboxylic_acid, arene — **no alkane**: TPA has no CH₂/CH₃ groups, only aromatic C-H, so the alkane SMARTS `[CX4;H3,H2]` does not match |
| EG (ethylene glycol) | alcohol, alkane |

This is the actual hydrolysis use case: the model should separate PET (ester)
from its hydrolysis products TPA (carboxylic_acid) and EG (alcohol).

In [ ]:
from sklearn.metrics import f1_score, classification_report

EXPECTED_GROUPS = {
    "PET": ["ester", "alkane", "arene", "ether"],
    "TPA": ["carboxylic_acid", "arene"],
    "EG":  ["alcohol", "alkane"],
}

y_true = np.array([
    [int(name in EXPECTED_GROUPS[p]) for name in LABEL_NAMES]
    for p in meta["Polymer"]
])

print(f"Micro F1:   {f1_score(y_true, y_pred, average='micro',   zero_division=0):.4f}")
print(f"Macro F1:   {f1_score(y_true, y_pred, average='macro',   zero_division=0):.4f}")
print(f"Samples F1: {f1_score(y_true, y_pred, average='samples', zero_division=0):.4f}")

hydro_idx = [LABEL_NAMES.index(n) for n in HYDROLYSIS_NAMES]
print(f"Hydrolysis macro F1: {f1_score(y_true[:, hydro_idx], y_pred[:, hydro_idx], average='macro', zero_division=0):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))

# Per-substance subset accuracy (all 12 labels exactly right)
exact = (y_pred == y_true).all(axis=1)
print("Exact-match rate per substance:")
print(pd.Series(exact).groupby(meta["Polymer"].values).mean().round(3).to_string())

## 6. Hydrolysis groups focus

For PET hydrolysis screening the four relevant groups are ester, carboxylic_acid, alcohol, arene.

In [ ]:
hydro = results.groupby("polymer")[HYDROLYSIS_NAMES].mean().round(3)

fig, ax = plt.subplots(figsize=(8, 4))
hydro.T.plot(kind="bar", ax=ax)
ax.set_ylabel("Fraction predicted positive")
ax.set_title("Hydrolysis functional groups — lab samples")
ax.set_ylim(0, 1.05)
ax.legend(title="Polymer", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 7. Prediction confidence per class

Raw probabilities (before thresholding), boxplot per polymer.

In [ ]:
probs = pd.DataFrame(y_prob, columns=LABEL_NAMES)
probs["polymer"] = meta["Polymer"].values

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, label in enumerate(LABEL_NAMES):
    ax = axes[i]
    groups = [probs[probs["polymer"] == p][label].values for p in sorted(probs["polymer"].unique())]
    ax.boxplot(groups, labels=sorted(probs["polymer"].unique()), patch_artist=True)
    ax.axhline(best_thresholds[i], color="red", linestyle="--", linewidth=0.8, label=f"t={best_thresholds[i]:.2f}")
    ax.set_title(label)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=7)

plt.suptitle("Prediction probabilities per class (red dashed = threshold)", y=1.01)
plt.tight_layout()
plt.show()


## 8. Sample spectra check

Plot a few raw preprocessed spectra to confirm they look reasonable.

In [ ]:
WAVENUMBERS = np.arange(600, 3900, 2)

substances = sorted(meta["Polymer"].unique())
fig, axes = plt.subplots(1, len(substances), figsize=(5 * len(substances), 4))
for ax, substance in zip(np.atleast_1d(axes), substances):
    idx = meta[meta["Polymer"] == substance].index[0]
    ax.plot(WAVENUMBERS, X_lab[idx], linewidth=0.8)
    ax.invert_xaxis()
    ax.set_title(substance)
    ax.set_xlabel("Wavenumber (cm⁻¹)")
    ax.set_ylabel("Normalised absorbance")

plt.suptitle("One example spectrum per substance (preprocessed)")
plt.tight_layout()
plt.show()
